# Synthèse transversale finale
## Conception, comparaison et analyse critique de modèles de deep learning

**Projet de fin de module — Deep Learning — EMSI 2025–2026**

---

Ce notebook conclut le projet. Il **relie** les trois parties (MLP / CNN / RNN-Seq2Seq) et répond à
la **question transversale finale** du cahier des charges. Il suppose que les trois notebooks
précédents ont été exécutés.


## 1. Récapitulatif des trois parties

| Partie | Données | Structure des données | Architecture | Biais inductif exploité | Métrique principale | Résultat obtenu* |
|--------|---------|------------------------|--------------|--------------------------|---------------------|------------------|
| **I — MLP** | Wine Quality (tabulaire) | Vecteur de variables, **sans** géométrie ni ordre | MLP (couches denses) | Aucun a priori spatial/temporel | Accuracy / F1 | acc ≈ **0,73**, F1 ≈ 0,72 |
| **II — CNN** | CIFAR-10 (images) | Grille 2D, **localité** + invariance par translation | CNN type LeNet amélioré | Localité, partage des poids, hiérarchie | Accuracy | CNN ≈ **0,64** vs MLP ≈ 0,46 |
| **III — Seq2Seq** | Tatoeba fra-eng (texte) | Suite ordonnée de longueur variable, **temporalité** | Encodeur–décodeur GRU | Récurrence, mémoire, conditionnement | Perplexité / BLEU | PPL ≈ **11,5**, BLEU(beam) ≈ 0,038 |

\* Résultats mesurés sur CPU avec sous-échantillonnage configurable (cf. notebooks). Les **tendances**
comparatives, et non les valeurs absolues, constituent le résultat scientifique.

**Constat central.** À chaque famille de données correspond une **architecture dont le biais inductif
épouse la structure statistique** des données. Quand on applique la « mauvaise » architecture (un MLP
sur des images, Partie II), la performance s'effondre malgré un nombre de paramètres comparable —
preuve expérimentale directe que **l'adéquation architecture ↔ structure des données** prime sur la
seule capacité du modèle.


## 2. Fil conducteur théorique commun

Les trois parties partagent **un même paradigme d'apprentissage supervisé** :
1. un modèle paramétré $f_\theta$ implémenté comme `nn.Module` ;
2. une **propagation avant** produisant des représentations puis une prédiction ;
3. une **fonction de perte** différentiable (entropie croisée, éventuellement masquée) ;
4. une **rétropropagation** (autograd) calculant $\nabla_\theta \mathcal{L}$ ;
5. une **mise à jour** par un optimiseur (Adam/SGD).

Ce qui change d'une partie à l'autre n'est **pas** le principe d'optimisation, mais la **façon dont
l'architecture transforme l'entrée en représentation** — c'est-à-dire les **hypothèses structurelles**
injectées dans $f_\theta$ :

- **MLP** : combinaisons linéaires globales + non-linéarités. Aucune hypothèse sur l'organisation des
  variables → adapté quand il n'y a **ni géométrie ni ordre** à exploiter (tabulaire).
- **CNN** : convolutions locales à poids partagés. Hypothèses de **localité** et d'**invariance par
  translation** → adapté à la **géométrie spatiale** des images. Les couches profondes composent une
  **hiérarchie** de représentations (bords → motifs → objets).
- **RNN/LSTM/GRU/Seq2Seq** : état caché récurrent transmis dans le temps. Hypothèse de **dépendance
  séquentielle** et de **mémoire** → adapté à la **temporalité** et aux **longueurs variables** du texte.
  Le schéma encodeur–décodeur ajoute le **conditionnement** d'une séquence cible sur une séquence source.


## 3. Question transversale finale

> *Comment le deep learning adapte-t-il ses architectures à la structure des données — tabulaire,
> image et séquentielle — et pourquoi un même paradigme d'apprentissage supervisé doit-il être décliné
> différemment selon la géométrie, la dépendance locale, la temporalité et la représentation des
> données ?*

**Réponse argumentée.**

Le deep learning repose sur un **noyau commun** — un modèle différentiable optimisé par descente de
gradient via rétropropagation — mais sa **puissance pratique** vient de l'**injection de biais
inductifs** adaptés à la **structure des données**. Apprendre, ce n'est pas seulement disposer de
capacité ; c'est disposer de la **bonne capacité, contrainte de la bonne manière**.

1. **Géométrie (images).** Les pixels voisins sont corrélés et un motif garde son sens où qu'il
   apparaisse. Le CNN encode directement cette **localité** (champ réceptif) et cette **invariance**
   (partage des poids), ce qui réduit drastiquement le nombre de paramètres *et* améliore la
   généralisation. La Partie II le montre : à paramètres comparables, le CNN dépasse largement le MLP.
   Un MLP devrait **réapprendre** l'invariance pour chaque position — gaspillage statistique que la
   structure convolutionnelle évite par construction.

2. **Dépendance locale vs globale.** La notion de « voisinage utile » diffère selon les données :
   spatial pour les images (pooling, stride contrôlent la résolution — relié aux calculs
   dimensionnels de la Partie II), temporel pour les séquences (l'état caché agrège le passé). Le
   tabulaire, lui, n'a **pas** de voisinage privilégié : d'où la pertinence d'un MLP « agnostique »,
   et la difficulté à battre des modèles à arbres qui modélisent des interactions de variables.

3. **Temporalité (séquences).** L'ordre porte le sens et les longueurs varient. Les RNN factorisent
   $P(x_{1:T})=\prod_t P(x_t\mid x_{<t})$ et mémorisent le contexte dans $h_t$. Mais la BPTT souffre de
   gradients **évanescents/explosifs** (illustré Partie III via le gradient clipping) ; les **portes**
   des LSTM/GRU créent un chemin stable pour l'information. Le **Seq2Seq** étend ce cadre à la
   **génération conditionnelle** (traduction), affinée par **beam search**.

4. **Représentation.** Chaque architecture impose une **représentation interne** alignée sur les
   données : vecteur dense (MLP), cartes de caractéristiques spatiales (CNN), état caché temporel
   (RNN). Le choix de représentation *est* le choix du biais inductif.

**Conclusion.** Un même paradigme supervisé doit être **décliné différemment** parce que la
**structure statistique** des données — géométrie, dépendance locale, temporalité, représentation —
détermine quelles invariances et quelles contraintes rendent l'apprentissage **efficace en données**
et **généralisable**. Le rôle de l'ingénieur en deep learning est précisément de **diagnostiquer la
structure des données** puis de **choisir (ou concevoir) l'architecture dont le biais inductif y
correspond** — démarche que ce projet a illustrée de bout en bout, du MLP tabulaire au Seq2Seq
récurrent, en passant par le CNN visuel.

> *Ouverture.* Les mécanismes d'**attention** et les **Transformers** poussent cette logique plus loin :
> ils remplacent le biais de localité/récurrence par un biais d'**alignement appris**, ce qui lève le
> goulot d'étranglement du contexte unique observé en Partie III — prolongement naturel de ce travail.


## 4. Bilan méthodologique et limites globales

- **Reproductibilité** : graines fixées, configurations centralisées (`CONFIG`), données téléchargées
  et mises en cache par les notebooks — aucun chemin ni hyperparamètre codé en dur hors config.
- **Honnêteté expérimentale** : exécution **CPU** avec sous-échantillonnage → les accuracies/BLEU
  absolus sont inférieurs à l'état de l'art, mais les **comparaisons** (MLP vs CNN, RNN vs LSTM/GRU,
  glouton vs beam) sont **contrôlées** (mêmes données, mêmes budgets) et donc valides.
- **Pistes** : passage à pleine échelle sur GPU (augmenter les paramètres de config), validation
  croisée (Partie I), augmentation de données et réseaux plus profonds (Partie II), mécanisme
  d'attention (Partie III).
